In [1]:
# --------------------------------------------------------------------------------
# 📦 Cell 1 – Install / Imports
# --------------------------------------------------------------------------------
# If you're in a fresh environment you may need to uncomment the pip lines once.
# !pip install torchvision torch tqdm albumentations==1.4.3 opencv-python-headless
# !pip install timm  # For EfficientNet and other models

import os
import copy
import random
from pathlib import Path
import numpy as np
import pandas as pd
from PIL import Image
import cv2
from scipy.ndimage import gaussian_filter, map_coordinates

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import transforms as T, models

import albumentations as A
from albumentations.pytorch import ToTensorV2
from matplotlib import pyplot as plt
from tqdm.auto import tqdm

# Set random seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)
random.seed(42)

In [2]:
# --------------------------------------------------------------------------------
# 📂 Cell 2 – Paths & Hyper-parameters (Enhanced)
# --------------------------------------------------------------------------------
DATA_CSV   = r"C:\Users\yozev\PycharmProjects\finetuning\combined_shape16_scores.csv"
IMG_DIR    = Path(r"C:\Users\yozev\OneDrive\Desktop\Shapes2\shape16")
REF_PATH   = IMG_DIR / "original16.png"

# Enhanced training setup
BATCH_SIZE     = 24          # Smaller batch for larger models
NUM_EPOCHS     = 35
BASE_LR        = 1e-4        # Lower base learning rate
WEIGHT_DECAY   = 2e-4        # Higher weight decay
VAL_FRAC       = 0.15
RANDOM_SEED    = 42
UNFREEZE_EPOCH = 8           # Later unfreezing for stability
WARMUP_EPOCHS  = 3           # Learning rate warmup
MIN_SAMPLES_PER_CLASS = 400  # More balanced data
DEVICE         = "cuda" if torch.cuda.is_available() else "cpu"

print("Device:", DEVICE)
print("PyTorch version:", torch.__version__)

Device: cpu
PyTorch version: 2.7.0+cpu


In [3]:
# --------------------------------------------------------------------------------
# 🖼️ Cell 3 – Advanced Augmentation Utilities
# --------------------------------------------------------------------------------
def dilate_white_line(img: np.ndarray, ksize: int = 3, iterations: int = 1):
    kernel = np.ones((ksize, ksize), np.uint8)
    return cv2.dilate(img, kernel, iterations=iterations)

class AdvancedAugmentations:
    """Sophisticated augmentations for child drawings."""

    @staticmethod
    def elastic_transform(image, alpha=30, sigma=4):
        """Elastic deformation to simulate drawing variations."""
        image_np = np.array(image)
        shape = image_np.shape

        dx = gaussian_filter((np.random.rand(*shape) * 2 - 1), sigma) * alpha
        dy = gaussian_filter((np.random.rand(*shape) * 2 - 1), sigma) * alpha

        x, y = np.meshgrid(np.arange(shape[1]), np.arange(shape[0]))
        indices = np.reshape(y + dy, (-1, 1)), np.reshape(x + dx, (-1, 1))

        try:
            distorted = map_coordinates(image_np, indices, order=1, mode='constant')
            return Image.fromarray(distorted.reshape(shape).astype(np.uint8), mode='L')
        except:
            return image

    @staticmethod
    def line_width_variation(image, factor_range=(0.7, 1.4)):
        """Vary line thickness to simulate different drawing pressure."""
        factor = random.uniform(*factor_range)
        image_np = np.array(image)

        if factor > 1.0:
            # Thicken lines
            kernel_size = max(2, int(factor * 2))
            kernel = np.ones((kernel_size, kernel_size), np.uint8)
            result = cv2.dilate(image_np, kernel, iterations=1)
        else:
            # Thin lines
            kernel = np.ones((2, 2), np.uint8)
            result = cv2.erode(image_np, kernel, iterations=1)

        return Image.fromarray(result, mode='L')

    @staticmethod
    def noise_and_imperfection(image, noise_prob=0.3, gap_prob=0.15):
        """Add realistic drawing imperfections."""
        image_np = np.array(image).copy()

        # Add small gaps in lines
        if random.random() < gap_prob:
            line_pixels = np.where(image_np > 100)
            if len(line_pixels[0]) > 0:
                n_remove = min(len(line_pixels[0]) // 30, 8)
                if n_remove > 0:
                    indices = np.random.choice(len(line_pixels[0]), n_remove, replace=False)
                    image_np[line_pixels[0][indices], line_pixels[1][indices]] = 0

        # Add small noise spots
        if random.random() < noise_prob:
            noise_pixels = np.random.choice(
                image_np.size, size=max(1, image_np.size // 300), replace=False
            )
            noise_coords = np.unravel_index(noise_pixels, image_np.shape)
            image_np[noise_coords] = np.random.choice([0, 255], size=len(noise_pixels))

        return Image.fromarray(image_np, mode='L')

class BoldLineTransform:
    def __init__(self, ksize=3, iterations=1):
        self.ksize = ksize
        self.iterations = iterations

    def __call__(self, img):
        arr = np.array(img, dtype=np.uint8)
        kernel = np.ones((self.ksize, self.ksize), np.uint8)
        dilated = cv2.dilate(arr, kernel, iterations=self.iterations)
        return Image.fromarray(dilated, mode="L")

class ZoomShiftTransform:
    def __init__(self, zoom_range=(0.8, 0.95), shift_range=0.12):
        self.zoom_range = zoom_range
        self.shift_range = shift_range

    def __call__(self, img):
        w, h = img.size
        zoom = random.uniform(*self.zoom_range)
        new_w, new_h = int(w * zoom), int(h * zoom)

        zoomed = img.resize((new_w, new_h), Image.Resampling.LANCZOS)
        result = Image.new('L', (w, h), 0)

        max_shift_x = int(w * self.shift_range)
        max_shift_y = int(h * self.shift_range)
        shift_x = random.randint(-max_shift_x, max_shift_x)
        shift_y = random.randint(-max_shift_y, max_shift_y)

        paste_x = max(0, min((w - new_w) // 2 + shift_x, w - new_w))
        paste_y = max(0, min((h - new_h) // 2 + shift_y, h - new_h))

        result.paste(zoomed, (paste_x, paste_y))
        return result

# Enhanced training transforms with progressive difficulty
enhanced_train_transforms = T.Compose([
    T.RandomApply([T.Lambda(AdvancedAugmentations.elastic_transform)], p=0.35),
    T.RandomApply([T.Lambda(AdvancedAugmentations.line_width_variation)], p=0.45),
    T.RandomApply([T.Lambda(AdvancedAugmentations.noise_and_imperfection)], p=0.25),
    T.RandomAffine(degrees=8, translate=(0.08, 0.08), scale=(0.88, 1.12), fill=0),
    T.RandomPerspective(distortion_scale=0.08, p=0.3),
    T.RandomApply([BoldLineTransform()], p=0.4),
    T.RandomApply([ZoomShiftTransform()], p=0.3),
    T.ToTensor(),
    T.RandomErasing(scale=(0.005, 0.025), ratio=(0.3, 3.3), value=0, p=0.3),
    T.Normalize([0.5], [0.5])
])

child_val_tf = T.Compose([
    T.ToTensor(),
    T.Normalize([0.5], [0.5])
])
ref_tf = child_val_tf


In [4]:
# --------------------------------------------------------------------------------
# 🗄️ Cell 4 – Massively Balanced Augmented Dataset
# --------------------------------------------------------------------------------
class MegaAugmentedDataset(Dataset):
    """
    Dataset that creates heavily augmented balanced samples across all score classes.
    Uses 10+ different augmentation strategies per sample.
    """
    NAME_TEMPLATE = "{id}.png"

    def __init__(self, csv_path, img_dir, ref_path, min_samples_per_class=400,
                 max_augmentations_per_sample=8, is_val=False):
        df = pd.read_csv(csv_path)
        self.img_dir = Path(img_dir)
        self.ref_path = Path(ref_path)
        self.is_val = is_val
        self.min_samples_per_class = min_samples_per_class
        self.max_augmentations_per_sample = max_augmentations_per_sample

        assert "child_id" in df.columns, "CSV missing 'child_id'"
        assert "score" in df.columns, "CSV missing 'score'"

        # Filter out missing images
        df["img_path"] = df["child_id"].apply(
            lambda cid: self.img_dir / self.NAME_TEMPLATE.format(id=cid)
        )
        missing_mask = ~df["img_path"].apply(Path.exists)
        n_missing = missing_mask.sum()
        if n_missing:
            print(f"⚠️  Skipping {n_missing} rows with missing images.")

        self.original_data = df[~missing_mask].reset_index(drop=True)

        # Define augmentation strategies
        self.augmentation_strategies = [
            ('original', lambda x: x),
            ('bold_lines', BoldLineTransform(ksize=3, iterations=1)),
            ('bold_lines_heavy', BoldLineTransform(ksize=4, iterations=2)),
            ('rotate_left', lambda x: x.rotate(-5, fillcolor=0)),
            ('rotate_right', lambda x: x.rotate(5, fillcolor=0)),
            ('rotate_slight_left', lambda x: x.rotate(-2, fillcolor=0)),
            ('rotate_slight_right', lambda x: x.rotate(2, fillcolor=0)),
            ('zoom_in', ZoomShiftTransform(zoom_range=(0.75, 0.85))),
            ('zoom_out', ZoomShiftTransform(zoom_range=(0.95, 1.05))),
            ('zoom_shift', ZoomShiftTransform(zoom_range=(0.85, 0.95), shift_range=0.15)),
            ('elastic', AdvancedAugmentations.elastic_transform),
            ('line_variation', AdvancedAugmentations.line_width_variation),
            ('imperfections', AdvancedAugmentations.noise_and_imperfection),
        ]

        if not is_val:
            self.balanced_data = self._create_mega_balanced_dataset()
            print(f"📊 Created mega-balanced dataset with {len(self.balanced_data)} samples")
            self._print_class_distribution()
        else:
            self.balanced_data = self.original_data.copy()
            self.balanced_data['augmentation'] = 'original'

    def _create_mega_balanced_dataset(self):
        """Create heavily balanced dataset using extensive augmentations."""
        score_column = self.original_data['score'] - 1
        class_counts = score_column.value_counts().sort_index()

        print("Original class distribution:")
        for class_idx, count in class_counts.items():
            print(f"  Score {class_idx + 1}: {count} samples")

        balanced_samples = []

        for class_idx in range(7):  # scores 1-7 (0-6 in 0-based)
            class_data = self.original_data[score_column == class_idx]
            current_count = len(class_data)

            if current_count == 0:
                print(f"⚠️  No samples found for score {class_idx + 1}")
                continue

            # Add original samples
            for _, row in class_data.iterrows():
                balanced_samples.append({
                    'child_id': row['child_id'],
                    'score': row['score'],
                    'img_path': row['img_path'],
                    'augmentation': 'original'
                })

            # Calculate augmented samples needed
            samples_needed = max(0, self.min_samples_per_class - current_count)

            if samples_needed > 0:
                augmentations_per_original = min(
                    self.max_augmentations_per_sample,
                    (samples_needed // current_count) + 1
                )

                for _, row in class_data.iterrows():
                    # Create multiple augmented versions
                    aug_strategies = random.sample(
                        self.augmentation_strategies[1:],  # Exclude 'original'
                        min(len(self.augmentation_strategies) - 1, augmentations_per_original)
                    )

                    for aug_name, _ in aug_strategies:
                        if len([x for x in balanced_samples if x['score'] == row['score']]) >= self.min_samples_per_class:
                            break

                        balanced_samples.append({
                            'child_id': row['child_id'],
                            'score': row['score'],
                            'img_path': row['img_path'],
                            'augmentation': aug_name
                        })

        return pd.DataFrame(balanced_samples)

    def _print_class_distribution(self):
        """Print the final class distribution."""
        score_counts = (self.balanced_data['score'] - 1).value_counts().sort_index()
        print("\nFinal mega-balanced class distribution:")
        total_samples = 0
        for class_idx, count in score_counts.items():
            print(f"  Score {class_idx + 1}: {count} samples")
            total_samples += count
        print(f"  Total: {total_samples} samples")

    def __len__(self):
        return len(self.balanced_data)

    def __getitem__(self, idx):
        row = self.balanced_data.iloc[idx]
        img_path = row["img_path"]
        score = int(row["score"]) - 1  # 1-7 → 0-6
        augmentation = row["augmentation"]

        # Load images
        child_img = Image.open(img_path).convert("L")
        ref_img = Image.open(self.ref_path).convert("L")

        # Apply specific augmentation if not validation
        if not self.is_val and augmentation != 'original':
            aug_func = next((func for name, func in self.augmentation_strategies if name == augmentation), None)
            if aug_func:
                try:
                    child_img = aug_func(child_img)
                except:
                    pass  # Keep original if augmentation fails

        # Apply final transforms
        if self.is_val:
            child_tensor = child_val_tf(child_img)
        else:
            child_tensor = enhanced_train_transforms(child_img)

        ref_tensor = ref_tf(ref_img)

        return {
            "child": child_tensor,
            "reference": ref_tensor,
            "label": torch.tensor(score, dtype=torch.long),
        }

In [5]:
# --------------------------------------------------------------------------------
# 🧠 Cell 5 – Multi-Scale Attention Siamese Architecture
# --------------------------------------------------------------------------------
class MultiScaleAttentionSiamese(nn.Module):
    """
    Enhanced architecture with:
    - EfficientNet backbone for better feature extraction
    - Multi-scale feature extraction
    - Cross-attention between child/reference
    - Geometric feature emphasis
    """
    def __init__(self, backbone_name="efficientnet_b1", num_classes=7, freeze=True):
        super().__init__()

        # Use EfficientNet for better feature extraction
        if backbone_name == "efficientnet_b1":
            try:
                backbone = models.efficientnet_b1(weights="IMAGENET1K_V1")
                backbone.features[0][0] = nn.Conv2d(1, 32, 3, 2, 1, bias=False)
                self.backbone = backbone.features
                feature_dim = 1280  # EfficientNet-B1 feature dim
            except:
                # Fallback to ResNet if EfficientNet not available
                print("EfficientNet not available, using ResNet50")
                backbone = models.resnet50(weights="IMAGENET1K_V1")
                backbone.conv1 = nn.Conv2d(1, 64, 7, 2, 3, bias=False)
                self.backbone = nn.Sequential(*list(backbone.children())[:-2])
                feature_dim = 2048
        else:
            backbone = models.__dict__[backbone_name](weights="IMAGENET1K_V1")
            backbone.conv1 = nn.Conv2d(1, 64, 7, 2, 3, bias=False)
            self.backbone = nn.Sequential(*list(backbone.children())[:-2])
            feature_dim = 512 if 'resnet18' in backbone_name else 2048

        # Freeze/unfreeze backbone
        for p in self.backbone.parameters():
            p.requires_grad = not freeze

        # Multi-scale feature extraction
        self.global_pool = nn.AdaptiveAvgPool2d((1, 1))
        self.regional_pool = nn.AdaptiveAvgPool2d((2, 2))
        self.local_pool = nn.AdaptiveAvgPool2d((4, 4))

        # Feature dimension calculation
        total_pooled_dim = feature_dim + feature_dim * 4 + feature_dim * 16  # 1x1 + 2x2 + 4x4

        # Cross-attention mechanism (simplified)
        self.attention_dim = min(256, feature_dim // 4)
        self.query_proj = nn.Linear(total_pooled_dim, self.attention_dim)
        self.key_proj = nn.Linear(total_pooled_dim, self.attention_dim)
        self.value_proj = nn.Linear(total_pooled_dim, self.attention_dim)

        # Comparison and classification
        comparison_dim = self.attention_dim * 2 + total_pooled_dim * 2

        self.classifier = nn.Sequential(
            nn.Linear(comparison_dim, 1024),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(1024, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.4),
            nn.Linear(512, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(256, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(0.2),
            nn.Linear(128, num_classes),
        )

    def extract_multiscale_features(self, x):
        """Extract features at multiple scales."""
        backbone_features = self.backbone(x)  # [B, C, H, W]

        # Multi-scale pooling
        global_feat = self.global_pool(backbone_features).flatten(1)      # [B, C]
        regional_feat = self.regional_pool(backbone_features).flatten(1)  # [B, C*4]
        local_feat = self.local_pool(backbone_features).flatten(1)        # [B, C*16]

        # Concatenate all scales
        multi_scale_features = torch.cat([global_feat, regional_feat, local_feat], dim=1)
        return multi_scale_features

    def apply_cross_attention(self, child_features, ref_features):
        """Apply simplified cross-attention between child and reference features."""
        # Project to attention space
        child_q = self.query_proj(child_features)
        ref_k = self.key_proj(ref_features)
        ref_v = self.value_proj(ref_features)

        # Compute attention scores
        attention_scores = torch.matmul(child_q.unsqueeze(1), ref_k.unsqueeze(2)).squeeze()
        attention_weights = F.softmax(attention_scores.unsqueeze(1), dim=-1)

        # Apply attention
        attended_features = attention_weights * ref_v

        return attended_features

    def forward(self, child, reference):
        # Extract multi-scale features
        child_features = self.extract_multiscale_features(child)
        ref_features = self.extract_multiscale_features(reference)

        # Apply cross-attention
        child_attended = self.apply_cross_attention(child_features, ref_features)
        ref_attended = self.apply_cross_attention(ref_features, child_features)

        # Combine all features
        combined_features = torch.cat([
            child_features, ref_features, child_attended, ref_attended
        ], dim=1)

        return self.classifier(combined_features)


In [6]:
# --------------------------------------------------------------------------------
# 🎯 Cell 6 – Advanced Loss Functions and Optimization
# --------------------------------------------------------------------------------
class ToleranceFocalLoss(nn.Module):
    """
    Advanced loss combining tolerance and focal loss for handling:
    - Hard examples (focal loss)
    - Class imbalance
    - Human rater variance (tolerance)
    """
    def __init__(self, alpha=1.0, gamma=2.0, tolerance=1, label_smoothing=0.1):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.tolerance = tolerance
        self.label_smoothing = label_smoothing

    def forward(self, outputs, targets):
        # Calculate predictions
        _, predicted = outputs.max(1)

        # Apply tolerance mask
        within_tolerance = torch.abs(predicted - targets) <= self.tolerance

        # Label smoothing
        log_probs = F.log_softmax(outputs, dim=1)
        targets_one_hot = torch.zeros_like(log_probs).scatter_(1, targets.unsqueeze(1), 1)

        if self.label_smoothing > 0:
            targets_smooth = targets_one_hot * (1 - self.label_smoothing) + self.label_smoothing / outputs.size(1)
        else:
            targets_smooth = targets_one_hot

        # Standard cross-entropy loss
        ce_loss = -(targets_smooth * log_probs).sum(dim=1)

        # Focal loss components
        pt = torch.exp(-ce_loss)
        focal_loss = self.alpha * (1 - pt) ** self.gamma * ce_loss

        # Apply tolerance mask (zero out losses for predictions within tolerance)
        adjusted_losses = focal_loss * (~within_tolerance).float()

        return adjusted_losses.mean()

class WarmupCosineScheduler:
    """Learning rate scheduler with warmup and cosine decay."""
    def __init__(self, optimizer, warmup_epochs, total_epochs, base_lr, min_lr=1e-6):
        self.optimizer = optimizer
        self.warmup_epochs = warmup_epochs
        self.total_epochs = total_epochs
        self.base_lr = base_lr
        self.min_lr = min_lr
        self.current_epoch = 0

    def step(self):
        if self.current_epoch < self.warmup_epochs:
            # Warmup phase
            lr = self.base_lr * (self.current_epoch + 1) / self.warmup_epochs
        else:
            # Cosine decay phase
            progress = (self.current_epoch - self.warmup_epochs) / (self.total_epochs - self.warmup_epochs)
            lr = self.min_lr + (self.base_lr - self.min_lr) * 0.5 * (1 + np.cos(np.pi * progress))

        for param_group in self.optimizer.param_groups:
            param_group['lr'] = lr

        self.current_epoch += 1
        return lr

# Mixup augmentation for additional regularization
def mixup_data(x, y, alpha=0.4):
    """Mixup augmentation."""
    if alpha > 0:
        lam = np.random.beta(alpha, alpha)
    else:
        lam = 1

    batch_size = x.size(0)
    index = torch.randperm(batch_size).to(x.device)

    mixed_x = lam * x + (1 - lam) * x[index]
    y_a, y_b = y, y[index]
    return mixed_x, y_a, y_b, lam

def mixup_criterion(criterion, pred, y_a, y_b, lam):
    """Mixup loss calculation."""
    return lam * criterion(pred, y_a) + (1 - lam) * criterion(pred, y_b)

In [7]:
# --------------------------------------------------------------------------------
# 🧹 Cell 7 – Enhanced Data Loading with Massive Augmentation
# --------------------------------------------------------------------------------
print("Creating massively augmented datasets...")

# Create full dataset for splitting
temp_ds = MegaAugmentedDataset(DATA_CSV, IMG_DIR, REF_PATH,
                              min_samples_per_class=1, is_val=True)

# Split original data indices
val_sz = int(len(temp_ds.original_data) * VAL_FRAC)
train_sz = len(temp_ds.original_data) - val_sz

train_indices, val_indices = torch.utils.data.random_split(
    range(len(temp_ds.original_data)),
    [train_sz, val_sz],
    generator=torch.Generator().manual_seed(RANDOM_SEED),
)

# Create separate DataFrames
train_df = temp_ds.original_data.iloc[train_indices.indices].reset_index(drop=True)
val_df = temp_ds.original_data.iloc[val_indices.indices].reset_index(drop=True)

# Save temporary CSV files
train_csv_path = "temp_train_mega.csv"
val_csv_path = "temp_val_mega.csv"
train_df.to_csv(train_csv_path, index=False)
val_df.to_csv(val_csv_path, index=False)

# Create mega-augmented training dataset
print("Creating mega-augmented training dataset...")
train_ds = MegaAugmentedDataset(
    train_csv_path, IMG_DIR, REF_PATH,
    min_samples_per_class=MIN_SAMPLES_PER_CLASS,
    max_augmentations_per_sample=8,
    is_val=False
)

print("\nCreating validation dataset...")
val_ds = MegaAugmentedDataset(
    val_csv_path, IMG_DIR, REF_PATH,
    min_samples_per_class=1,
    is_val=True
)

# Create data loaders
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE,
                         shuffle=True, num_workers=0, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE,
                       shuffle=False, num_workers=0, pin_memory=True)

print(f"\n📊 Final dataset sizes:")
print(f"Training samples: {len(train_ds)}")
print(f"Validation samples: {len(val_ds)}")

# Clean up temporary files
try:
    os.remove(train_csv_path)
    os.remove(val_csv_path)
except:
    pass

Creating massively augmented datasets...
⚠️  Skipping 2 rows with missing images.
Creating mega-augmented training dataset...
Original class distribution:
  Score 1: 89 samples
  Score 2: 117 samples
  Score 3: 161 samples
  Score 4: 178 samples
  Score 5: 138 samples
  Score 6: 72 samples
  Score 7: 7 samples
📊 Created mega-balanced dataset with 2463 samples

Final mega-balanced class distribution:
  Score 1: 400 samples
  Score 2: 400 samples
  Score 3: 400 samples
  Score 4: 400 samples
  Score 5: 400 samples
  Score 6: 400 samples
  Score 7: 63 samples
  Total: 2463 samples

Creating validation dataset...

📊 Final dataset sizes:
Training samples: 2463
Validation samples: 134


In [8]:
# --------------------------------------------------------------------------------
# 🚀 Cell 8 – Model Initialization with Advanced Architecture
# --------------------------------------------------------------------------------
print("Initializing enhanced model architecture...")

model = MultiScaleAttentionSiamese(
    backbone_name="efficientnet_b1",
    num_classes=7,
    freeze=True
).to(DEVICE)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

# Advanced loss function
criterion = ToleranceFocalLoss(
    alpha=1.0,
    gamma=2.0,
    tolerance=1,
    label_smoothing=0.1
)

# Optimizer with advanced settings
optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=BASE_LR,
    weight_decay=WEIGHT_DECAY,
    betas=(0.9, 0.999),
    eps=1e-8
)

# Advanced learning rate scheduler
scheduler = WarmupCosineScheduler(
    optimizer,
    warmup_epochs=WARMUP_EPOCHS,
    total_epochs=NUM_EPOCHS,
    base_lr=BASE_LR,
    min_lr=1e-6
)

print("Model and training setup complete!")


Initializing enhanced model architecture...
Downloading: "https://download.pytorch.org/models/efficientnet_b1_rwightman-bac287d4.pth" to C:\Users\yozev/.cache\torch\hub\checkpoints\efficientnet_b1_rwightman-bac287d4.pth


100%|██████████| 30.1M/30.1M [00:04<00:00, 7.86MB/s]


Total parameters: 83,422,695
Trainable parameters: 76,910,087
Model and training setup complete!


In [9]:
# --------------------------------------------------------------------------------
# 🔁 Cell 9 – Enhanced Training & Validation Functions
# --------------------------------------------------------------------------------
def calculate_tolerant_accuracy(outputs, targets, tolerance=1):
    """Calculate accuracy considering predictions within tolerance as correct."""
    _, predicted = outputs.max(1)
    within_tolerance = torch.abs(predicted - targets) <= tolerance
    return within_tolerance.float().mean().item()

def train_one_epoch_enhanced(model, loader, criterion, optimizer, use_mixup=True):
    """Enhanced training with mixup and advanced metrics."""
    model.train()
    running_loss, exact_correct, tolerant_correct, total = 0, 0, 0, 0

    for batch in tqdm(loader, desc="Training", leave=False):
        child = batch["child"].to(DEVICE, non_blocking=True)
        ref = batch["reference"].to(DEVICE, non_blocking=True)
        labels = batch["label"].to(DEVICE, non_blocking=True)

        # Apply mixup augmentation
        if use_mixup and random.random() < 0.3:  # 30% chance of mixup
            child, labels_a, labels_b, lam = mixup_data(child, labels, alpha=0.4)

            optimizer.zero_grad()
            outputs = model(child, ref)
            loss = mixup_criterion(criterion, outputs, labels_a, labels_b, lam)
        else:
            optimizer.zero_grad()
            outputs = model(child, ref)
            loss = criterion(outputs, labels)

        loss.backward()

        # Gradient clipping for stability
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        optimizer.step()

        running_loss += loss.item() * labels.size(0)
        _, preds = outputs.max(1)

        if use_mixup and random.random() < 0.3:
            # For mixup, use original labels for accuracy calculation
            exact_correct += preds.eq(labels_a).sum().item()
            tolerant_correct += (torch.abs(preds - labels_a) <= 1).sum().item()
        else:
            exact_correct += preds.eq(labels).sum().item()
            tolerant_correct += (torch.abs(preds - labels) <= 1).sum().item()

        total += labels.size(0)

    return running_loss / total, exact_correct / total, tolerant_correct / total

@torch.no_grad()
def evaluate_enhanced(model, loader, criterion):
    """Enhanced evaluation with detailed metrics."""
    model.eval()
    running_loss, exact_correct, tolerant_correct, total = 0, 0, 0, 0
    all_preds, all_targets = [], []

    for batch in tqdm(loader, desc="Validation", leave=False):
        child = batch["child"].to(DEVICE, non_blocking=True)
        ref = batch["reference"].to(DEVICE, non_blocking=True)
        labels = batch["label"].to(DEVICE, non_blocking=True)

        outputs = model(child, ref)
        loss = criterion(outputs, labels)

        running_loss += loss.item() * labels.size(0)
        _, preds = outputs.max(1)
        exact_correct += preds.eq(labels).sum().item()
        tolerant_correct += (torch.abs(preds - labels) <= 1).sum().item()
        total += labels.size(0)

        all_preds.extend(preds.cpu().numpy())
        all_targets.extend(labels.cpu().numpy())

    return running_loss / total, exact_correct / total, tolerant_correct / total, all_preds, all_targets


In [10]:
# --------------------------------------------------------------------------------
# 🎯 Cell 10 – Enhanced Training Loop with Early Stopping
# --------------------------------------------------------------------------------
print("Starting enhanced training with early stopping...")

history = {
    "train_loss": [], "val_loss": [],
    "train_exact_acc": [], "val_exact_acc": [],
    "train_tolerant_acc": [], "val_tolerant_acc": [],
    "learning_rates": []
}

best_tolerant_val = 0.0
best_model_state = None
patience = 8
patience_counter = 0
min_delta = 0.005  # Minimum improvement to reset patience

print("Training Progress:")
print("=" * 80)

for epoch in range(1, NUM_EPOCHS + 1):
    # Learning rate scheduling
    current_lr = scheduler.step()

    # Unfreeze backbone after specified epoch
    if epoch == UNFREEZE_EPOCH:
        print(f"\n🔓 Unfreezing backbone at epoch {epoch}")
        for p in model.backbone.parameters():
            p.requires_grad = True

        # Re-create optimizer to include backbone params
        optimizer = torch.optim.AdamW(
            model.parameters(),
            lr=BASE_LR * 0.1,  # Lower LR for backbone
            weight_decay=WEIGHT_DECAY,
            betas=(0.9, 0.999)
        )

        # Reset scheduler for remaining epochs
        scheduler = WarmupCosineScheduler(
            optimizer,
            warmup_epochs=2,
            total_epochs=NUM_EPOCHS - UNFREEZE_EPOCH + 1,
            base_lr=BASE_LR * 0.1,
            min_lr=1e-7
        )
        current_lr = scheduler.step()

    # Training phase
    tr_loss, tr_exact, tr_tolerant = train_one_epoch_enhanced(
        model, train_loader, criterion, optimizer, use_mixup=(epoch > 3)
    )

    # Validation phase
    val_loss, val_exact, val_tolerant, val_preds, val_targets = evaluate_enhanced(
        model, val_loader, criterion
    )

    # Update history
    history["train_loss"].append(tr_loss)
    history["val_loss"].append(val_loss)
    history["train_exact_acc"].append(tr_exact)
    history["val_exact_acc"].append(val_exact)
    history["train_tolerant_acc"].append(tr_tolerant)
    history["val_tolerant_acc"].append(val_tolerant)
    history["learning_rates"].append(current_lr)

    # Check for best model
    improvement = val_tolerant - best_tolerant_val
    is_best = improvement > min_delta

    if is_best:
        best_tolerant_val = val_tolerant
        best_model_state = model.state_dict().copy()
        patience_counter = 0
        best_indicator = "  🎯 NEW BEST!"
    else:
        patience_counter += 1
        best_indicator = ""

    # Print progress
    print(f"Epoch {epoch:02d}/{NUM_EPOCHS} | "
          f"Loss: {tr_loss:.3f}/{val_loss:.3f} | "
          f"Exact: {tr_exact:.3f}/{val_exact:.3f} | "
          f"Tolerant: {tr_tolerant:.3f}/{val_tolerant:.3f} | "
          f"LR: {current_lr:.2e}"
          f"{best_indicator}")

    # Early stopping check
    if patience_counter >= patience:
        print(f"\n⏹️  Early stopping triggered after {patience} epochs without improvement")
        print(f"🏆 Best tolerant validation accuracy: {best_tolerant_val:.4f}")
        break

# Load best model
if best_model_state is not None:
    model.load_state_dict(best_model_state)
    print(f"\n✅ Loaded best model with tolerant accuracy: {best_tolerant_val:.4f}")

print("\nTraining completed!")


Starting enhanced training with early stopping...
Training Progress:


C:\Users\yozev\AppData\Roaming\Python\Python311\site-packages\torch\utils\data\dataloader.py:665: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Training:   0%|          | 0/103 [00:00<?, ?it/s]

Validation:   0%|          | 0/6 [00:00<?, ?it/s]

Epoch 01/35 | Loss: 0.791/0.590 | Exact: 0.168/0.224 | Tolerant: 0.441/0.560 | LR: 3.33e-05  🎯 NEW BEST!


...

Training:   0%|          | 0/103 [00:00<?, ?it/s]

Validation:   0%|          | 0/6 [00:00<?, ?it/s]

Epoch 04/35 | Loss: 0.745/0.574 | Exact: 0.165/0.224 | Tolerant: 0.434/0.560 | LR: 1.00e-04


Training:   0%|          | 0/103 [00:00<?, ?it/s]

UnboundLocalError: cannot access local variable 'labels_a' where it is not associated with a value

In [ ]:
# --------------------------------------------------------------------------------
# 📊 Cell 11 – Advanced Results Analysis and Visualization
# --------------------------------------------------------------------------------
print("Analyzing results...")

# Create comprehensive plots
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
fig.suptitle('Enhanced Model Training Analysis', fontsize=16, fontweight='bold')

# Loss curves
axes[0, 0].plot(history["train_loss"], label="Train Loss", linewidth=2)
axes[0, 0].plot(history["val_loss"], label="Val Loss", linewidth=2)
axes[0, 0].set_title("Loss Curves")
axes[0, 0].set_xlabel("Epoch")
axes[0, 0].set_ylabel("Loss")
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# Exact accuracy curves
axes[0, 1].plot(history["train_exact_acc"], label="Train Exact", linewidth=2)
axes[0, 1].plot(history["val_exact_acc"], label="Val Exact", linewidth=2)
axes[0, 1].set_title("Exact Accuracy")
axes[0, 1].set_xlabel("Epoch")
axes[0, 1].set_ylabel("Accuracy")
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# Tolerant accuracy curves
axes[0, 2].plot(history["train_tolerant_acc"], label="Train Tolerant (±1)", linewidth=2, color='green')
axes[0, 2].plot(history["val_tolerant_acc"], label="Val Tolerant (±1)", linewidth=2, color='orange')
axes[0, 2].set_title("Tolerant Accuracy (±1)")
axes[0, 2].set_xlabel("Epoch")
axes[0, 2].set_ylabel("Accuracy")
axes[0, 2].legend()
axes[0, 2].grid(True, alpha=0.3)
axes[0, 2].axhline(y=0.9, color='red', linestyle='--', alpha=0.7, label='90% Target')

# Learning rate schedule
axes[1, 0].plot(history["learning_rates"], linewidth=2, color='purple')
axes[1, 0].set_title("Learning Rate Schedule")
axes[1, 0].set_xlabel("Epoch")
axes[1, 0].set_ylabel("Learning Rate")
axes[1, 0].set_yscale('log')
axes[1, 0].grid(True, alpha=0.3)

# Confusion matrix for final validation predictions
from sklearn.metrics import confusion_matrix
import seaborn as sns

# Get final predictions
model.eval()
final_preds, final_targets = [], []
with torch.no_grad():
    for batch in val_loader:
        child = batch["child"].to(DEVICE)
        ref = batch["reference"].to(DEVICE)
        labels = batch["label"].to(DEVICE)
        outputs = model(child, ref)
        _, preds = outputs.max(1)
        final_preds.extend(preds.cpu().numpy())
        final_targets.extend(labels.cpu().numpy())

cm = confusion_matrix(final_targets, final_preds)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[1, 1])
axes[1, 1].set_title("Confusion Matrix")
axes[1, 1].set_xlabel("Predicted Score")
axes[1, 1].set_ylabel("True Score")

# Prediction distribution
score_distribution = pd.DataFrame({
    'True Score': [i+1 for i in final_targets],
    'Predicted Score': [i+1 for i in final_preds]
})

axes[1, 2].hist([score_distribution['True Score'], score_distribution['Predicted Score']],
                bins=7, alpha=0.7, label=['True', 'Predicted'],
                color=['blue', 'red'], edgecolor='black')
axes[1, 2].set_title("Score Distribution")
axes[1, 2].set_xlabel("Score")
axes[1, 2].set_ylabel("Frequency")
axes[1, 2].legend()
axes[1, 2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


In [ ]:
# --------------------------------------------------------------------------------
# 📈 Cell 12 – Detailed Performance Metrics
# --------------------------------------------------------------------------------
from sklearn.metrics import classification_report, mean_absolute_error

print("=" * 80)
print("FINAL PERFORMANCE ANALYSIS")
print("=" * 80)

# Convert back to 1-7 scale for reporting
final_targets_1to7 = [t + 1 for t in final_targets]
final_preds_1to7 = [p + 1 for p in final_preds]

# Calculate various metrics
exact_accuracy = sum(t == p for t, p in zip(final_targets, final_preds)) / len(final_targets)
tolerant_accuracy = sum(abs(t - p) <= 1 for t, p in zip(final_targets, final_preds)) / len(final_targets)
mae = mean_absolute_error(final_targets_1to7, final_preds_1to7)

print(f"📊 ACCURACY METRICS:")
print(f"   Exact Accuracy:     {exact_accuracy:.4f} ({exact_accuracy*100:.2f}%)")
print(f"   Tolerant Accuracy:  {tolerant_accuracy:.4f} ({tolerant_accuracy*100:.2f}%)")
print(f"   Mean Absolute Error: {mae:.4f}")
print(f"   Best Training Tolerant Acc: {max(history['train_tolerant_acc']):.4f}")

print(f"\n🎯 TARGET ACHIEVEMENT:")
target_achieved = "✅ ACHIEVED!" if tolerant_accuracy >= 0.9 else "❌ Not yet achieved"
print(f"   90% Tolerant Accuracy: {target_achieved}")

print(f"\n📈 IMPROVEMENT ANALYSIS:")
initial_val_tolerant = history['val_tolerant_acc'][0] if history['val_tolerant_acc'] else 0
improvement = tolerant_accuracy - initial_val_tolerant
print(f"   Initial Tolerant Accuracy: {initial_val_tolerant:.4f}")
print(f"   Final Tolerant Accuracy:   {tolerant_accuracy:.4f}")
print(f"   Total Improvement:         +{improvement:.4f} ({improvement*100:.2f}%)")

# Detailed classification report
print(f"\n📋 DETAILED CLASSIFICATION REPORT:")
print("=" * 60)
print(classification_report(final_targets_1to7, final_preds_1to7,
                          target_names=[f'Score {i}' for i in range(1, 8)],
                          zero_division=0))

# Per-class tolerant accuracy
print(f"\n🎯 PER-CLASS TOLERANT ACCURACY:")
print("=" * 40)
for score in range(1, 8):
    true_indices = [i for i, t in enumerate(final_targets_1to7) if t == score]
    if true_indices:
        tolerant_correct = sum(abs(final_targets_1to7[i] - final_preds_1to7[i]) <= 1 for i in true_indices)
        class_tolerant_acc = tolerant_correct / len(true_indices)
        print(f"   Score {score}: {class_tolerant_acc:.4f} ({class_tolerant_acc*100:.2f}%) - {len(true_indices)} samples")

In [ ]:
# --------------------------------------------------------------------------------
# 💾 Cell 13 – Model Saving and Export
# --------------------------------------------------------------------------------
MODEL_PATH = "enhanced_shape16_similarity_model.pt"
CHECKPOINT_PATH = "enhanced_shape16_checkpoint.pt"

print(f"\n💾 SAVING MODEL...")

# Save the complete model state
torch.save({
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'epoch': epoch,
    'best_tolerant_accuracy': best_tolerant_val,
    'history': history,
    'model_config': {
        'backbone_name': 'efficientnet_b1',
        'num_classes': 7,
        'tolerance': 1
    }
}, CHECKPOINT_PATH)

# Save just the model for inference
torch.save(model.state_dict(), MODEL_PATH)

print(f"✅ Model saved to: {MODEL_PATH}")
print(f"✅ Full checkpoint saved to: {CHECKPOINT_PATH}")


In [ ]:
# --------------------------------------------------------------------------------
# 🔮 Cell 14 – Inference Function for New Images
# --------------------------------------------------------------------------------
def predict_similarity_score(model, child_image_path, reference_image_path, device=DEVICE):
    """
    Predict similarity score for a new child drawing.

    Args:
        model: Trained model
        child_image_path: Path to child drawing
        reference_image_path: Path to reference image
        device: Device to run inference on

    Returns:
        predicted_score: Score from 1-7
        confidence: Model confidence (softmax probability)
    """
    model.eval()

    # Load and preprocess images
    child_img = Image.open(child_image_path).convert("L")
    ref_img = Image.open(reference_image_path).convert("L")

    # Apply validation transforms
    child_tensor = child_val_tf(child_img).unsqueeze(0).to(device)
    ref_tensor = ref_tf(ref_img).unsqueeze(0).to(device)

    with torch.no_grad():
        outputs = model(child_tensor, ref_tensor)
        probabilities = F.softmax(outputs, dim=1)
        predicted_class = outputs.argmax(1).item()
        confidence = probabilities[0, predicted_class].item()

    # Convert back to 1-7 scale
    predicted_score = predicted_class + 1

    return predicted_score, confidence

# Example usage function
def batch_predict_scores(model, image_paths, reference_path):
    """Predict scores for multiple images."""
    results = []

    print("Predicting similarity scores...")
    for img_path in tqdm(image_paths):
        try:
            score, confidence = predict_similarity_score(model, img_path, reference_path)
            results.append({
                'image_path': img_path,
                'predicted_score': score,
                'confidence': confidence
            })
        except Exception as e:
            print(f"Error processing {img_path}: {e}")
            results.append({
                'image_path': img_path,
                'predicted_score': None,
                'confidence': None
            })

    return results

print(f"\n🎉 TRAINING COMPLETE!")
print(f"📊 Final Results Summary:")
print(f"   • Exact Accuracy: {exact_accuracy*100:.2f}%")
print(f"   • Tolerant Accuracy: {tolerant_accuracy*100:.2f}%")
print(f"   • Model saved and ready for inference!")


In [ ]:
# --------------------------------------------------------------------------------
# 🚀 Cell 15 – Optional: Ensemble Training for Even Higher Accuracy
# --------------------------------------------------------------------------------
def train_ensemble_models(n_models=3, base_model_path=MODEL_PATH):
    """
    Train multiple models with different initializations for ensemble prediction.
    This can push accuracy even higher by combining multiple model predictions.
    """
    print(f"\n🔄 Training ensemble of {n_models} models...")

    ensemble_models = []
    ensemble_accuracies = []

    for i in range(n_models):
        print(f"\n📚 Training ensemble model {i+1}/{n_models}")

        # Set different random seeds for diversity
        torch.manual_seed(42 + i * 123)
        np.random.seed(42 + i * 123)
        random.seed(42 + i * 123)

        # Create model with slight variations
        model_i = MultiScaleAttentionSiamese(
            backbone_name="efficientnet_b1",
            num_classes=7,
            freeze=True
        ).to(DEVICE)

        # Slightly different hyperparameters for diversity
        lr_variations = [BASE_LR * 0.8, BASE_LR, BASE_LR * 1.2]
        wd_variations = [WEIGHT_DECAY * 0.5, WEIGHT_DECAY, WEIGHT_DECAY * 1.5]

        optimizer_i = torch.optim.AdamW(
            filter(lambda p: p.requires_grad, model_i.parameters()),
            lr=lr_variations[i],
            weight_decay=wd_variations[i]
        )

        # Train for fewer epochs since we already know good hyperparameters
        # (You would implement a simplified training loop here)
        # For brevity, we'll just save the current best model as ensemble member
        ensemble_models.append(model.state_dict().copy())
        ensemble_accuracies.append(tolerant_accuracy)

    return ensemble_models, ensemble_accuracies

def ensemble_predict(ensemble_models, child, reference, device=DEVICE):
    """Predict using ensemble of models."""
    predictions = []

    for model_state in ensemble_models:
        # Create temporary model and load state
        temp_model = MultiScaleAttentionSiamese(num_classes=7).to(device)
        temp_model.load_state_dict(model_state)
        temp_model.eval()

        with torch.no_grad():
            output = temp_model(child, reference)
            prob = F.softmax(output, dim=1)
            predictions.append(prob)

    # Average predictions
    ensemble_pred = torch.stack(predictions).mean(dim=0)
    return ensemble_pred

# Uncomment the following lines if you want to train an ensemble
# ensemble_models, ensemble_accs = train_ensemble_models(n_models=3)
# print(f"Ensemble trained with individual accuracies: {ensemble_accs}")

print(f"\n🎯 NOTEBOOK COMPLETE!")
print(f"💡 Tips for even higher accuracy:")
print(f"   • Train ensemble of 3-5 models")
print(f"   • Collect more training data")
print(f"   • Try Vision Transformer backbone")
print(f"   • Experiment with test-time augmentation")
print(f"   • Fine-tune on domain-specific augmentation